In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix

# Create results directory if it doesn't exist
os.makedirs('results', exist_ok=True)

# 1. LOAD DATASET
df = pd.read_csv("customer_churn_nn.csv")

# 2. EXPLORE DATA
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nMissing values per column:\n", df.isnull().sum())

# 3. DATA PREPROCESSING
X = df.drop(columns=['customer_id', 'churn'])
y = df['churn']

categorical_cols = ['region', 'plan_type', 'contract_type', 'payment_method']
numerical_cols = [col for col in X.columns if col not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_cols)
    ])

X_processed = preprocessor.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42, stratify=y)

# 4. BUILD NEURAL NETWORK MODEL
baseline_model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(16, activation='relu'),   # Hidden layer
    Dense(8, activation='relu'),    # Hidden layer
    Dense(1, activation='sigmoid')  # Output layer
])

baseline_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
                       loss='binary_crossentropy',
                       metrics=['accuracy'])

# 5. TRAIN MODEL
print("\n--- Training Baseline Model ---")
history = baseline_model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=30, batch_size=32, verbose=1)

# 6. SAVE EVALUATION CHARTS
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history.history['loss'], label='Train Loss')
ax1.plot(history.history['val_loss'], label='Test Loss')
ax1.set_title('Model Loss Over Epochs')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.legend()

ax2.plot(history.history['accuracy'], label='Train Accuracy')
ax2.plot(history.history['val_accuracy'], label='Test Accuracy')
ax2.set_title('Model Accuracy Over Epochs')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Accuracy')
ax2.legend()
plt.tight_layout()
plt.savefig('results/evaluation_outputs.png')
plt.close()

# 7. EXPERIMENTATION LOGIC
def experiment(layers, lr, batch_s, epochs_n):
    m = Sequential()
    m.add(Input(shape=(X_train.shape[1],)))
    for neurons in layers:
        m.add(Dense(neurons, activation='relu'))
    m.add(Dense(1, activation='sigmoid'))
    m.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr), loss='binary_crossentropy', metrics=['accuracy'])
    m.fit(X_train, y_train, epochs=epochs_n, batch_size=batch_s, verbose=0)
    _, t_acc = m.evaluate(X_train, y_train, verbose=0)
    _, v_acc = m.evaluate(X_test, y_test, verbose=0)
    return t_acc, v_acc

print("\n--- Running Hyperparameter Experiments ---")
t1, v1 = experiment([8], 0.001, 64, 20)
t2, v2 = experiment([32, 16, 8], 0.005, 32, 40)
t3, v3 = experiment([16, 16], 0.1, 128, 25)

results_log = [
    {"Config": "Baseline (16->8 units)", "LR": 0.01, "Batch Size": 32, "Epochs": 30, "Train Acc": history.history['accuracy'][-1], "Test Acc": history.history['val_accuracy'][-1]},
    {"Config": "Exp 1: Shallow (8 units)", "LR": 0.001, "Batch Size": 64, "Epochs": 20, "Train Acc": t1, "Test Acc": v1},
    {"Config": "Exp 2: Deep (32->16->8 units)", "LR": 0.005, "Batch Size": 32, "Epochs": 40, "Train Acc": t2, "Test Acc": v2},
    {"Config": "Exp 3: Extreme LR (16->16 units)", "LR": 0.1, "Batch Size": 128, "Epochs": 25, "Train Acc": t3, "Test Acc": v3}
]

comparison_df = pd.DataFrame(results_log)
comparison_df.to_csv("results/model_comparison_table.csv", index=False)
print("\n--- Model Comparison Table Saved ---")
print(comparison_df.to_string(index=False))

Dataset Shape: 2000 rows, 17 columns

Missing values per column:
 customer_id                     0
region                          0
plan_type                       0
contract_type                   0
payment_method                  0
tenure_months                   0
monthly_charges_inr             0
avg_login_days_per_month        0
support_tickets_last_90_days    0
payment_delay_days              0
data_usage_gb                   0
satisfaction_score              0
last_complaint_days_ago         0
discount_percent                0
autopay_enabled                 0
referral_count                  0
churn                           0
dtype: int64

--- Training Baseline Model ---
Epoch 1/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9794 - loss: 0.1761 - val_accuracy: 0.9850 - val_loss: 0.0668
Epoch 2/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9844 - loss: 0.0734 - val_accuracy: 0.9850 - val_loss: 0.0507
Epoch 3/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accura